In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import os
import sys

# Subsampling

In [2]:
from running.tl import subsample_from_annote

raw_data_repo = '../data/'
adtfile = 'HLCA_core.h5ad'

# obsfile = 'HLCA_core_obs.csv'
# obs = pd.read_csv(os.path.join(raw_data_repo, obsfile), index_col=0)
adata = sc.read_h5ad(os.path.join(raw_data_repo, adtfile))
adata.X = adata.raw.X
obs = adata.obs

c:\Users\CGX\AppData\Roaming\mamba\envs\topicvi\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# level2 'Myeloid' / finest
# level2 'Alveolar epithelium' -- finest
# level3 'basal'
# level3 'Dendritic cells'
# level3 'Fibroblasts'
# level3 'T cell lineage'
# level3 'Secretory' -- finest
# level2 'Lymphoid' -- finest

In [6]:
obs[['ann_level_3', 'ann_level_4', 'ann_level_5']].value_counts().sort_index()

ann_level_3                   ann_level_4                ann_level_5                  
AT1                           None                       None                              7937
AT2                           AT2 proliferating          None                               976
                              None                       None                             61429
B cell lineage                B cells                    None                              4511
                              Plasma cells               None                              1773
Basal                         Basal resting              None                             38955
                              Hillock-like               None                              4600
                              Suprabasal                 None                             41158
Dendritic cells               DC1                        None                               322
                              DC2                

In [14]:
# 'ann_finest_level', 'ann_level_1'
sample_cells = dict()
sample_cells['Myeloid_l2'] = subsample_from_annote(obs, 'ann_level_2', 'Myeloid', n_cells=5000, sample_strategy='balanced', next_level=None)
sample_cells['Myeloid_lf'] = subsample_from_annote(obs, 'ann_level_2', 'Myeloid', n_cells=5000, next_level='ann_finest_level')
sample_cells['Basal_l3'] = subsample_from_annote(obs, 'ann_level_3', 'Basal', n_cells=5000, sample_strategy='balanced', next_level=None)
sample_cells['DC_l3'] = subsample_from_annote(obs, 'ann_level_3', 'Dendritic cells', n_cells=1200, sample_strategy='balanced', next_level=None)
sample_cells['Fibroblasts_l3'] = subsample_from_annote(obs, 'ann_level_3', 'Fibroblasts', n_cells=5000, next_level=None)
sample_cells['T_l3'] = subsample_from_annote(obs, 'ann_level_3', 'T cell lineage', n_cells=1500, sample_strategy='balanced', next_level=None)
sample_cells['Secretory_lf'] = subsample_from_annote(obs, 'ann_level_3', 'Secretory', n_cells=5000, next_level='ann_finest_level')
sample_cells['Lymphoid_lf'] = subsample_from_annote(obs, 'ann_level_2', 'Lymphoid', n_cells=5000, next_level='ann_finest_level')

Subsample 1250 cells from each ann_level_3, total 5000 cells.
Subsample 1666 cells from each ann_level_4, total 4998 cells.
Subsample 300 cells from each ann_level_4, total 1200 cells.
Subsample 500 cells from each ann_level_4, total 1500 cells.


In [18]:
saving_dir = '../data/HLCA_subsamped'
os.makedirs(saving_dir, exist_ok=True)

for k, v in sample_cells.items():
    print(k)
    save_p = os.path.join(saving_dir, f"HLCA_{k}.h5ad")
    adata[v, :].write_h5ad(save_p)

Myeloid_l2
Myeloid_lf
Basal_l3
DC_l3
Fibroblasts_l3
T_l3
Secretory_lf
Lymphoid_lf


In [26]:
del adata

In [25]:
!ls -l -h $saving_dir

total 1016M
-rw-r--r-- 1 u12219016 u12219016 239M Nov  2 11:23 HLCA_Basal_l3.h5ad
-rw-r--r-- 1 u12219016 u12219016  38M Nov  2 11:23 HLCA_DC_l3.h5ad
-rw-r--r-- 1 u12219016 u12219016 168M Nov  2 11:23 HLCA_Fibroblasts_l3.h5ad
-rw-r--r-- 1 u12219016 u12219016  82M Nov  2 11:23 HLCA_Lymphoid_lf.h5ad
-rw-r--r-- 1 u12219016 u12219016 129M Nov  2 11:23 HLCA_Myeloid_l2.h5ad
-rw-r--r-- 1 u12219016 u12219016 166M Nov  2 11:23 HLCA_Myeloid_lf.h5ad
-rw-r--r-- 1 u12219016 u12219016 157M Nov  2 11:23 HLCA_Secretory_lf.h5ad
-rw-r--r-- 1 u12219016 u12219016  40M Nov  2 11:23 HLCA_T_l3.h5ad


# Preprocess [HVG]

In [ ]:
from running.tl import load_config, preprocess_adata
import yaml

In [2]:
raw_data_repo = '/groups/zjupgx_zz/home/u12219016/Cai/data_repo'
save_data_repo = '../results'
os.makedirs(save_data_repo, exist_ok=True)
os.listdir(raw_data_repo)

['Lung_atlas.h5ad',
 'Human_pancreas.h5ad',
 'GSE262072_CART_Processed_compress.h5ad',
 'tcga_pancan.h5ad',
 'pbmc10k.h5ad',
 'pbmc1.h5ad',
 'Pancan_T_atals_integrated_full.h5ad',
 'zheng68k.h5ad',
 'zheng68k_sorted.h5ad',
 'GSE219098_data.h5mu',
 'HLCA_core.h5ad',
 'Mariana_24NC.h5ad',
 'Bassez.h5ad',
 'zenodo_7761954.h5ad',
 'HLCA_core_obs.csv']

In [3]:
for file in os.listdir(raw_data_repo):
    print(file)
    if not file.endswith('h5ad'): continue
    if 'tcga' in file: continue
    if file in ['HLCA_core.h5ad','GSE262072_CART_Processed_compress.h5ad']:
        continue
    data_name = file.replace('.h5ad', '')
    
    project_dir = os.path.join(save_data_repo, data_name)
    os.makedirs(project_dir, exist_ok=True)
    if os.path.exists(os.path.join(project_dir, 'adata.h5ad')):
        print("The data have been preprocessed. skip.")
        continue
    
    adata = sc.read_h5ad(
        os.path.join(raw_data_repo, file)
    )

    print(adata)
    print(adata.obs.head())
    
    batch_key = input("select batch key")
    if not batch_key:
        batch_key=None
    label_key = input("select label key")
    layer = input("select layer.")
    if layer:
        adata.X = adata.layers[layer]
        
    n_clusters = np.unique(adata.obs[label_key]).size
    n_topics = 2000 // n_clusters + 10
    
    default_config = load_config('./running_config.yaml')
    default_config['project_name'] = data_name
    default_config['save_dir'] = project_dir
    default_config['data_kwargs'] = {
        "batch_key": batch_key,
        "label_key": label_key, 
        "size_factor_key": "size_factor",
        "annotation_key": "annotation" 
    }
    default_config['model_kwargs'] = {
        "n_topics": n_topics,
        "n_clusters": n_clusters
    }
    
    with open(os.path.join(project_dir, 'running_config.yaml'), "w") as f:
        yaml.dump(default_config, f)
    
    pdata = preprocess_adata(adata, nhvg=2000, batch_key=batch_key)
    pdata.write(os.path.join(project_dir, 'adata.h5ad'))
    

Lung_atlas.h5ad
The data have been preprocessed. skip.
Human_pancreas.h5ad
The data have been preprocessed. skip.
GSE262072_CART_Processed_compress.h5ad
tcga_pancan.h5ad
pbmc10k.h5ad
The data have been preprocessed. skip.
pbmc1.h5ad
The data have been preprocessed. skip.
Pancan_T_atals_integrated_full.h5ad
The data have been preprocessed. skip.
zheng68k.h5ad
The data have been preprocessed. skip.
zheng68k_sorted.h5ad
The data have been preprocessed. skip.
GSE219098_data.h5mu
HLCA_core.h5ad
Mariana_24NC.h5ad
AnnData object with n_obs × n_vars = 391989 × 45575
    obs: 'nCount_RNA', 'nFeature_RNA', 'harm_study', 'harm_healthy.tissue', 'harm_tumor.site', 'harm_sample.type', 'harm_condition', 'harm_tumor.type', 'harm_cd45pos', 'harm_healthy.pat', 'percent.mt', 'ratio_nCount_nFeature', 'batch', 'X_scvi_batch', 'X_scvi_labels', 'X_scvi_local_l_mean', 'X_scvi_local_l_var', 'leiden_0.2', 'leiden_0.4', 'leiden_0.6', 'leiden_0.8', 'leiden_1', 'leiden_1.2', 'leiden_1.4', 'author_first_cell_type',

select batch key batch
select label key cell_type
select layer. 


Total number of cells: 391989
Number of cells after filtering of low quality cells: 375739
Error in highly_variable_genes 'seurat_v3', will try to run with default params: b'reciprocal condition number  2.1065e-16'


/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:478: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:478: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:478: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_var

Bassez.h5ad
AnnData object with n_obs × n_vars = 226635 × 22567
    obs: 'nCount_RNA', 'nFeature_RNA', 'patient_id', 'timepoint', 'expansion', 'BC_type', 'cellType', 'cohort'
                                  nCount_RNA  nFeature_RNA patient_id  \
BIOKEY_33_Pre_AAACCTGAGAGACTTA-1        3911          1665  BIOKEY_33   
BIOKEY_33_Pre_AAACCTGAGTAGCGGT-1         605           491  BIOKEY_33   
BIOKEY_33_Pre_AAACCTGCATGGTAGG-1         596           461  BIOKEY_33   
BIOKEY_33_Pre_AAACCTGGTATAGGGC-1        2983          1615  BIOKEY_33   
BIOKEY_33_Pre_AAACCTGGTCAGGACA-1        4098          1657  BIOKEY_33   

                                 timepoint expansion BC_type    cellType  \
BIOKEY_33_Pre_AAACCTGAGAGACTTA-1       Pre         E    TNBC      T_cell   
BIOKEY_33_Pre_AAACCTGAGTAGCGGT-1       Pre         E    TNBC  Fibroblast   
BIOKEY_33_Pre_AAACCTGCATGGTAGG-1       Pre         E    TNBC      T_cell   
BIOKEY_33_Pre_AAACCTGGTATAGGGC-1       Pre         E    TNBC  Fibroblast   
BIOKEY

select batch key patient_id
select label key cellType
select layer. 


Total number of cells: 226635
Number of cells after filtering of low quality cells: 144160
zenodo_7761954.h5ad
The data have been preprocessed. skip.
HLCA_core_obs.csv


In [96]:
raw_data_repo = '../data/HLCA_subsamped'
save_data_repo = '../results/subsampled'
os.makedirs(save_data_repo, exist_ok=True)
os.listdir(raw_data_repo)

['HLCA_DC_l3.h5ad',
 'HLCA_Secretory_lf.h5ad',
 'HLCA_Fibroblasts_l3.h5ad',
 'HLCA_Myeloid_lf.h5ad',
 'HLCA_T_l3.h5ad',
 'HLCA_Myeloid_l2.h5ad',
 'HLCA_Lymphoid_lf.h5ad',
 'HLCA_Basal_l3.h5ad']

In [107]:
file = 'HLCA_T_l3.h5ad'
print(file)

data_name = file.replace('.h5ad', '')
project_dir = os.path.join(save_data_repo, data_name)
os.makedirs(project_dir, exist_ok=True)
adata = sc.read_h5ad(
    os.path.join(raw_data_repo, file)
)

print(adata)

batch_key = 'study'
if data_name.endswith('l2'):
    label_key = 'ann_level_3'
elif data_name.endswith('l3'):
    label_key = 'ann_level_4'
elif data_name.endswith('lf'):
    label_key = 'ann_finest_level'
layer = ''
if layer:
    adata.X = adata.layers[layer]

n_clusters = np.unique(adata.obs[label_key]).size
n_topics = 2000 // n_clusters + 10

default_config = load_config('./running_config.yaml')
default_config['project_name'] = data_name
default_config['save_dir'] = project_dir
default_config['data_kwargs'] = {
    "batch_key": batch_key,
    "label_key": label_key, 
    "size_factor_key": "size_factor",
    "annotation_key": "annotation" 
}
default_config['model_kwargs'] = {
    "n_topics": n_topics,
    "n_clusters": n_clusters
}

with open(os.path.join(project_dir, 'running_config.yaml'), "w") as f:
    yaml.dump(default_config, f)

pdata = preprocess_adata(adata, nhvg=2000, batch_key=batch_key)
pdata.write(os.path.join(project_dir, 'adata.h5ad'))

HLCA_T_l3.h5ad
AnnData object with n_obs × n_vars = 1500 × 27957
    obs: 'suspension_type', 'donor_id', 'is_primary_data', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id', 'organism_ontology_term_id', 'sex_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling', 'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'cause_of_death', 'dataset', 'entropy_dataset_leiden_3', 'entropy_original_ann_level_1_leiden_3', 'entropy_original_ann_level_2_clean_leiden_3', 'entropy_original_ann_level_3_clean_leiden_3', 'entropy_subject_ID_leiden_3', 'fresh_or_frozen', 'leiden_1', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'log10_total_counts', 'lung_condition', 'mixed_ancestry', 'n_genes_detected', 'original_ann_highest_res', 'original_an

In [ ]:
from running.tl import load_config,preprocess_adata
import yaml

for file in os.listdir(raw_data_repo):
    print(file)
    if not file.endswith('h5ad'): continue
    if 'tcga' in file: continue
    data_name = file.replace('.h5ad', '')
    
    project_dir = os.path.join(save_data_repo, data_name)
    os.makedirs(project_dir, exist_ok=True)
    if os.path.exists(os.path.join(project_dir, 'adata.h5ad')):
        print("The data have been preprocessed. skip.")
        continue
    
    adata = sc.read_h5ad(
        os.path.join(raw_data_repo, file)
    )

    print(adata)
    
    batch_key = 'study'
    if data_name.endswith('l2'):
        label_key = 'ann_level_3'
    elif data_name.endswith('l3'):
        label_key = 'ann_level_4'
    elif data_name.endswith('lf'):
        label_key = 'ann_finest_level'
    layer = ''
    if layer:
        adata.X = adata.layers[layer]
        
    n_clusters = np.unique(adata.obs[label_key]).size
    n_topics = 2000 // n_clusters + 10
    
    default_config = load_config('./running_config.yaml')
    default_config['project_name'] = data_name
    default_config['save_dir'] = project_dir
    default_config['data_kwargs'] = {
        "batch_key": batch_key,
        "label_key": label_key, 
        "size_factor_key": "size_factor",
        "annotation_key": "annotation" 
    }
    default_config['model_kwargs'] = {
        "n_topics": n_topics,
        "n_clusters": n_clusters
    }
    
    with open(os.path.join(project_dir, 'running_config.yaml'), "w") as f:
        yaml.dump(default_config, f)
    
    pdata = preprocess_adata(adata, nhvg=2000, batch_key=batch_key)
    pdata.write(os.path.join(project_dir, 'adata.h5ad'))
    

HLCA_DC_l3.h5ad
AnnData object with n_obs × n_vars = 1200 × 27957
    obs: 'suspension_type', 'donor_id', 'is_primary_data', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id', 'organism_ontology_term_id', 'sex_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling', 'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'cause_of_death', 'dataset', 'entropy_dataset_leiden_3', 'entropy_original_ann_level_1_leiden_3', 'entropy_original_ann_level_2_clean_leiden_3', 'entropy_original_ann_level_3_clean_leiden_3', 'entropy_subject_ID_leiden_3', 'fresh_or_frozen', 'leiden_1', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'log10_total_counts', 'lung_condition', 'mixed_ancestry', 'n_genes_detected', 'original_ann_highest_res', 'original_a

/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:478: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:478: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:478: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_var

HLCA_Secretory_lf.h5ad
AnnData object with n_obs × n_vars = 5000 × 27957
    obs: 'suspension_type', 'donor_id', 'is_primary_data', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id', 'organism_ontology_term_id', 'sex_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling', 'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'cause_of_death', 'dataset', 'entropy_dataset_leiden_3', 'entropy_original_ann_level_1_leiden_3', 'entropy_original_ann_level_2_clean_leiden_3', 'entropy_original_ann_level_3_clean_leiden_3', 'entropy_subject_ID_leiden_3', 'fresh_or_frozen', 'leiden_1', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'log10_total_counts', 'lung_condition', 'mixed_ancestry', 'n_genes_detected', 'original_ann_highest_res', 'ori

/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:478: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:478: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:478: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_var

HLCA_Fibroblasts_l3.h5ad
AnnData object with n_obs × n_vars = 5000 × 27957
    obs: 'suspension_type', 'donor_id', 'is_primary_data', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id', 'organism_ontology_term_id', 'sex_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling', 'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'cause_of_death', 'dataset', 'entropy_dataset_leiden_3', 'entropy_original_ann_level_1_leiden_3', 'entropy_original_ann_level_2_clean_leiden_3', 'entropy_original_ann_level_3_clean_leiden_3', 'entropy_subject_ID_leiden_3', 'fresh_or_frozen', 'leiden_1', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'log10_total_counts', 'lung_condition', 'mixed_ancestry', 'n_genes_detected', 'original_ann_highest_res', 'o

...

HLCA_Myeloid_lf.h5ad
AnnData object with n_obs × n_vars = 5000 × 27957
    obs: 'suspension_type', 'donor_id', 'is_primary_data', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id', 'organism_ontology_term_id', 'sex_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling', 'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'cause_of_death', 'dataset', 'entropy_dataset_leiden_3', 'entropy_original_ann_level_1_leiden_3', 'entropy_original_ann_level_2_clean_leiden_3', 'entropy_original_ann_level_3_clean_leiden_3', 'entropy_subject_ID_leiden_3', 'fresh_or_frozen', 'leiden_1', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'log10_total_counts', 'lung_condition', 'mixed_ancestry', 'n_genes_detected', 'original_ann_highest_res', 'origi

/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:478: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:478: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:478: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_var

HLCA_T_l3.h5ad
AnnData object with n_obs × n_vars = 1500 × 27957
    obs: 'suspension_type', 'donor_id', 'is_primary_data', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id', 'organism_ontology_term_id', 'sex_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling', 'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'cause_of_death', 'dataset', 'entropy_dataset_leiden_3', 'entropy_original_ann_level_1_leiden_3', 'entropy_original_ann_level_2_clean_leiden_3', 'entropy_original_ann_level_3_clean_leiden_3', 'entropy_subject_ID_leiden_3', 'fresh_or_frozen', 'leiden_1', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'log10_total_counts', 'lung_condition', 'mixed_ancestry', 'n_genes_detected', 'original_ann_highest_res', 'original_an

/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:478: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:478: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_variable_genes.py:478: FutureWarning: The frame.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  hvg = hvg.append(missing_hvg, ignore_index=True)
/home/u12219016/.conda/envs/pytorch/lib/python3.11/site-packages/scanpy/preprocessing/_highly_var

HLCA_Basal_l3.h5ad
AnnData object with n_obs × n_vars = 4998 × 27957
    obs: 'suspension_type', 'donor_id', 'is_primary_data', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id', 'organism_ontology_term_id', 'sex_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling', 'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'cause_of_death', 'dataset', 'entropy_dataset_leiden_3', 'entropy_original_ann_level_1_leiden_3', 'entropy_original_ann_level_2_clean_leiden_3', 'entropy_original_ann_level_3_clean_leiden_3', 'entropy_subject_ID_leiden_3', 'fresh_or_frozen', 'leiden_1', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'log10_total_counts', 'lung_condition', 'mixed_ancestry', 'n_genes_detected', 'original_ann_highest_res', 'origina

# Add Annotation

- generate the prior knowledge
- include the background and clusters markers.
- then filtering the prior knowledge based on the var names.

In [ ]:
import sys
from topicvi import *
from running.tl import load_config
from prior import *

In [3]:
def clearn_prior_dict(prior: dict, adata, min_genes=5):
    prior = {k: [i for i in v if i in adata.var_names] for k, v in prior.items()}
    prior = {k: v for k, v in prior.items() if len(v) > min_genes}
    # print({k: len(v) for k, v in prior.items()})
    return prior

In [19]:
import yaml

def write_config(config, path='../running_config.yaml'):
    with open(path, "w") as f:
        yaml.dump(config, f)

def setting_config(adata, config):
    config['train_kwargs']['max_epochs'] = 500
    config['train_kwargs']['batch_size'] = 1024
    config['model_kwargs']['n_topics'] = adata.shape[1] // config['model_kwargs']['n_clusters'] // 10 + 10
    config['extra_kwargs']['amortized_lda'] = {'train_kwargs': dict(early_stopping_monitor = 'elbo_train')}
    config['extra_kwargs']['pycogaps'] = {'train_kwargs': dict(nIterations = 5000)}
    config['extra_kwargs']['spectra'] = {'data_kwargs': dict(label_key = None), 'train_kwargs': dict(use_gpu=False)}
    config['extra_kwargs']['topicvi'] = {
        'train_kwargs': dict(pretrain_model = os.path.join(config['save_dir'], 'topicvi', 'pretrain')),
        'data_kwargs': dict(label_key = None)
    }
    config['model_kwargs']['n_topics'] = config['model_kwargs']['n_clusters']*2 + max(10, adata.shape[1] // 200)
    if adata.shape[0] > 100000:
        config['extra_kwargs']['scanvi_seed_label'] = {
            'train_kwargs': dict(plan_kwargs = dict(lr=0.0001), max_epochs=200) # To avoid nan bugs. when training.
        }
    return config

In [9]:
# basis: ['Hallmark', 'Reactome']

basis = get_priors('Hallmark')
basis.update(get_priors('Reactome'))
basis = {k.replace('/', '_'):v for k,v in basis.items()} # for storing.
print("Background genesets: ",len(basis), 'items')

Background genesets:  1868 items


In [13]:
cellmarker_db = pd.read_excel('../src/data/CellMarkerDB.Human.xlsx')
cellmarker_db.head()

In [15]:
# from the tissue

sctype = load_sctype_db()
print("Valid tissue types:")
print('\t'.join(sctype['tissue_type'].unique().tolist()))
print("Valid cell types:")
print('\t'.join(sctype['cell_type'].unique().tolist()))

Valid tissue types:
Immune system	Pancreas	Liver	Eye	Kidney	Brain	Lung	Adrenal	Heart	Intestine	Muscle	Placenta	Spleen	Stomach	Thymus
Valid cell types:
Pro-B cells	Pre-B cells	Naive B cells	Memory B cells	Plasma B cells	Naive CD8+ T cells	Naive CD4+ T cells	Memory CD8+ T cells	Memory CD4+ T cells	Effector CD8+ T cells	Effector CD4+ T cells	γδ-T cells	Platelets	CD8+ NKT-like cells	CD4+ NKT-like cells	Natural killer  cells	Eosinophils	Neutrophils	Basophils	Mast cells	Classical Monocytes	Non-classical monocytes	Intermediate monocytes	Macrophages	Megakaryocyte	Endothelial	Erythroid-like and erythroid precursor cells	HSC/MPP cells	Progenitor cells	Myeloid Dendritic cells	Plasmacytoid Dendritic cells	Granulocytes	ISG expressing immune cells	Acinar cells	Alpha cells	Beta cells	Delta cells	Ductal cells	Endothelial cells	Epsilon cells	Gamma (PP) cells	Immune system cells	Mesenchymal cells	Pancreatic progenitor cells	Pancreatic stellate cells	Peri-islet Schwann cells	Cholangiocytes	Endothelial ce

In [16]:
save_dir = '../results'
os.listdir(save_dir)

['zenodo_7761954',
 'tcga_pancan',
 '.ipynb_checkpoints',
 'subsampled',
 'pbmc1',
 'GSE262072_CART_Processed_compress',
 'zheng68k',
 'pbmc10k',
 'Human_pancreas',
 'Mariana_24NC',
 'zheng68k_sorted',
 'Lung_atlas',
 'Pancan_T_atals_integrated_full',
 'Bassez']

### pbmc1

In [20]:
data_name = 'pbmc1'
project_dir = os.path.join(save_dir, data_name)

adata_dir = os.path.join(project_dir, 'adata.h5ad')
config_dir = os.path.join(project_dir, 'running_config.yaml')
config = load_config(config_dir)
label_key = config['data_kwargs']['label_key']
adata = sc.read_h5ad(adata_dir)
write_config(setting_config(adata, config), config_dir)

print("Original annote:")
print(adata.obs[label_key].unique())

tissue = 'Immune system'
sctype_res = search_sctype_db(
    cell_type='',
    tissue_type=tissue,
)
sctype_res = filter_prior_dict(
    sctype_res, keywords = ['B cells', 'T cells', 'Natural killer', 'Macrophages', 'Megakaryocyte', 'Monocytes', 'Dendritic']
)
print(sctype_res.keys())

cm_res = search_cellmarker_db(
    cell_types=['B cell','Megakaryocyte','CD8 T','Natural killer','Monocyte','Myeloid','CD4 T','Dendritic'],
    tissue_type='Peripheral blood',
    cancer_type='Normal',
)

data_specific = {**sctype_res, **cm_res}
data_specific = clearn_prior_dict(data_specific, adata, min_genes = 5)
backgrounds = clearn_prior_dict(basis, adata, min_genes = 10)

print("#####")
print(adata.shape)
print(len(data_specific), len(backgrounds))

adata.uns[config['data_kwargs']['annotation_key']] = {
    'background': backgrounds,
    'clusters': data_specific
}
adata.uns['running_config'] = config
adata.write_h5ad(adata_dir)

Original annote:
['CD4 T cells', 'CD8 T cells', 'NK cells', 'Megakaryocytes', 'B cells', 'Dendritic Cells', 'CD14+ Monocytes', 'FCGR3A+ Monocytes']
Categories (8, object): ['B cells', 'CD4 T cells', 'CD8 T cells', 'CD14+ Monocytes', 'Dendritic Cells', 'FCGR3A+ Monocytes', 'Megakaryocytes', 'NK cells']
Loading cell type database
Searching for tissue type Immune system
Searching for cell type 
dict_keys(['Pro-B cells', 'Pre-B cells', 'Naive B cells', 'Memory B cells', 'Plasma B cells', 'Naive CD8+ T cells', 'Naive CD4+ T cells', 'Memory CD8+ T cells', 'Memory CD4+ T cells', 'Effector CD8+ T cells', 'Effector CD4+ T cells', 'γδ-T cells', 'Natural killer  cells', 'Classical Monocytes', 'Macrophages', 'Megakaryocyte', 'Myeloid Dendritic cells', 'Plasmacytoid Dendritic cells'])
Searching for cell markers with tissue type Peripheral blood and cancer type Normal
#####
(13060, 2000)
22 249


### pbmc10k

In [21]:
data_name = 'pbmc10k'
project_dir = os.path.join(save_dir, data_name)

adata_dir = os.path.join(project_dir, 'adata.h5ad')
config_dir = os.path.join(project_dir, 'running_config.yaml')
config = load_config(config_dir)
label_key = config['data_kwargs']['label_key']
adata = sc.read_h5ad(adata_dir)
write_config(setting_config(adata, config), config_dir)

print("Original annote:")
print(adata.obs[label_key].unique())

tissue = 'Immune system'
sctype_res = search_sctype_db(
    cell_type='',
    tissue_type=tissue,
)


sctype_res = filter_prior_dict(
    sctype_res, keywords = ['B cells', 'T cells', 'Natural killer', 'Macrophages', 'Megakaryocyte', 'Monocytes', 'Dendritic']
)
print(sctype_res.keys())

cm_res = search_cellmarker_db(
    cell_types=['B cell','Megakaryocyte','CD8 T','Natural killer','Monocyte','Myeloid','CD4 T','Dendritic'],
    tissue_type='Peripheral blood',
    cancer_type='Normal',
)

data_specific = {**sctype_res, **cm_res}
data_specific = clearn_prior_dict(data_specific, adata, min_genes = 5)
backgrounds = clearn_prior_dict(basis, adata, min_genes = 10)

print("#####")
print(adata.shape)
print(len(data_specific), len(backgrounds))

adata.uns[config['data_kwargs']['annotation_key']] = {
    'background': backgrounds,
    'clusters': data_specific
}
adata.uns['running_config'] = config
adata.write_h5ad(adata_dir)

Original annote:
['CD4 T cells', 'CD14+ Monocytes', 'CD8 T cells', 'B cells', 'Other', 'Dendritic Cells', 'FCGR3A+ Monocytes', 'NK cells', 'Megakaryocytes']
Categories (9, object): ['B cells', 'CD4 T cells', 'CD8 T cells', 'CD14+ Monocytes', ..., 'FCGR3A+ Monocytes', 'Megakaryocytes', 'NK cells', 'Other']
Loading cell type database
Searching for tissue type Immune system
Searching for cell type 
dict_keys(['Pro-B cells', 'Pre-B cells', 'Naive B cells', 'Memory B cells', 'Plasma B cells', 'Naive CD8+ T cells', 'Naive CD4+ T cells', 'Memory CD8+ T cells', 'Memory CD4+ T cells', 'Effector CD8+ T cells', 'Effector CD4+ T cells', 'γδ-T cells', 'Natural killer  cells', 'Classical Monocytes', 'Macrophages', 'Megakaryocyte', 'Myeloid Dendritic cells', 'Plasmacytoid Dendritic cells'])
Searching for cell markers with tissue type Peripheral blood and cancer type Normal
#####
(11746, 669)
22 107


### Human_pancreas

In [22]:
data_name = 'Human_pancreas'
project_dir = os.path.join(save_dir, data_name)

adata_dir = os.path.join(project_dir, 'adata.h5ad')
config_dir = os.path.join(project_dir, 'running_config.yaml')
config = load_config(config_dir)
label_key = config['data_kwargs']['label_key']
adata = sc.read_h5ad(adata_dir)
write_config(setting_config(adata, config), config_dir)


print("Original annote:")
print(adata.obs[label_key].unique().tolist())

sctype_res = search_sctype_db(
    cell_type='',
    tissue_type='Pancreas',
)

cm_res = search_cellmarker_db(
    cell_types=[i.replace('_', ' ') for i in adata.obs[label_key].unique().tolist()]+['stellate'],
    tissue_type='Pancreas',
    cancer_type='Normal',
)
cm_res.keys()

data_specific = {**sctype_res, **cm_res}
data_specific = clearn_prior_dict(data_specific, adata, min_genes = 5)
backgrounds = clearn_prior_dict(basis, adata, min_genes = 10)

print("#####")
print(adata.shape)
print(len(data_specific), len(backgrounds))

adata.uns[config['data_kwargs']['annotation_key']] = {
    'background': backgrounds,
    'clusters': data_specific
}
adata.uns['running_config'] = config
adata.write_h5ad(adata_dir)

Original annote:
['gamma', 'acinar', 'alpha', 'delta', 'beta', 'ductal', 'endothelial', 'activated_stellate', 'schwann', 'mast', 'macrophage', 'epsilon', 'quiescent_stellate', 't_cell']
Loading cell type database
Searching for tissue type Pancreas
Searching for cell type 
Searching for cell markers with tissue type Pancreas and cancer type Normal


/home/u12219016/jupyterlab/deepbicluster/evaluate/../src/utils/prior.py:125: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  markers = pd.Series(markers).dropna().unique()


#####
(14590, 2000)
12 299


### zheng68k

In [23]:
data_name = 'zheng68k'
project_dir = os.path.join(save_dir, data_name)

adata_dir = os.path.join(project_dir, 'adata.h5ad')
config_dir = os.path.join(project_dir, 'running_config.yaml')
config = load_config(config_dir)
label_key = config['data_kwargs']['label_key']
adata = sc.read_h5ad(adata_dir)
write_config(setting_config(adata, config), config_dir)

print("Original annote:")
print(adata.obs[label_key].unique())

tissue = 'Immune system'
sctype_res = search_sctype_db(
    cell_type='',
    tissue_type=tissue,
)


cm_res = search_cellmarker_db(
    cell_types=[
    'B cells','Treg','Th','CD8+ T','Natural killer','Monocytes','Stem Cell','CD4+ T','Th0','Cytotoxic','Th17'
    ],
    tissue_type='Peripheral blood',
    cancer_type='Normal',
)

data_specific = {**sctype_res, **cm_res}
data_specific = clearn_prior_dict(data_specific, adata, min_genes = 5)
backgrounds = clearn_prior_dict(basis, adata, min_genes = 10)

print("#####")
print(adata.shape)
print(len(data_specific), len(backgrounds))

adata.uns[config['data_kwargs']['annotation_key']] = {
    'background': backgrounds,
    'clusters': data_specific
}
adata.uns['running_config'] = config
adata.write_h5ad(adata_dir)

Original annote:
['CD8+ Cytotoxic T', 'CD8+/CD45RA+ Naive Cytotoxic', 'CD4+/CD45RO+ Memory', 'CD19+ B', 'CD4+/CD25 T Reg', ..., 'CD4+ T Helper2', 'CD4+/CD45RA+/CD25- Naive T', 'CD34+', 'CD14+ Monocyte', 'Dendritic']
Length: 11
Categories (11, object): ['CD14+ Monocyte', 'CD19+ B', 'CD34+', 'CD4+ T Helper2', ..., 'CD56+ NK', 'CD8+ Cytotoxic T', 'CD8+/CD45RA+ Naive Cytotoxic', 'Dendritic']
Loading cell type database
Searching for tissue type Immune system
Searching for cell type 
Searching for cell markers with tissue type Peripheral blood and cancer type Normal


/home/u12219016/jupyterlab/deepbicluster/evaluate/../src/utils/prior.py:125: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  markers = pd.Series(markers).dropna().unique()


#####
(60905, 2000)
23 182


### zheng68k_sorted

In [24]:
data_name = 'zheng68k_sorted'
project_dir = os.path.join(save_dir, data_name)

adata_dir = os.path.join(project_dir, 'adata.h5ad')
config_dir = os.path.join(project_dir, 'running_config.yaml')
config = load_config(config_dir)
label_key = config['data_kwargs']['label_key']
adata = sc.read_h5ad(adata_dir)
write_config(setting_config(adata, config), config_dir)

print("Original annote:")
print(adata.obs[label_key].unique())

tissue = 'Immune system'
sctype_res = search_sctype_db(
    cell_type='',
    tissue_type=tissue,
)


cm_res = search_cellmarker_db(
    cell_types=[
    'B cells','Treg','Th','CD8+ T','Natural killer','Monocytes','Stem Cell','CD4+ T','Th0','Cytotoxic','Th17'
    ],
    tissue_type='Peripheral blood',
    cancer_type='Normal',
)


data_specific = {**sctype_res, **cm_res}
data_specific = clearn_prior_dict(data_specific, adata, min_genes = 5)
backgrounds = clearn_prior_dict(basis, adata, min_genes = 10)

print("#####")
print(adata.shape)
print(len(data_specific), len(backgrounds))

adata.uns[config['data_kwargs']['annotation_key']] = {
    'background': backgrounds,
    'clusters': data_specific
}
adata.uns['running_config'] = config
adata.write_h5ad(adata_dir)

Original annote:
['CD14+ Monocyte', 'CD19+ B', 'CD34+', 'CD4+ T Helper2', 'CD4+/CD25 T Reg', 'CD4+/CD45RA+/CD25- Naive T', 'CD4+/CD45RO+ Memory', 'CD56+ NK', 'CD8+ Cytotoxic T', 'CD8+/CD45RA+ Naive Cytotoxic']
Categories (10, object): ['CD4+ T Helper2', 'CD4+/CD25 T Reg', 'CD4+/CD45RA+/CD25- Naive T', 'CD4+/CD45RO+ Memory', ..., 'CD14+ Monocyte', 'CD19+ B', 'CD34+', 'CD56+ NK']
Loading cell type database
Searching for tissue type Immune system
Searching for cell type 
Searching for cell markers with tissue type Peripheral blood and cancer type Normal


/home/u12219016/jupyterlab/deepbicluster/evaluate/../src/utils/prior.py:125: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  markers = pd.Series(markers).dropna().unique()


#####
(19245, 2000)
26 237


### zenodo_7761954

In [25]:
data_name = 'zenodo_7761954'
project_dir = os.path.join(save_dir, data_name)

adata_dir = os.path.join(project_dir, 'adata.h5ad')
config_dir = os.path.join(project_dir, 'running_config.yaml')
config = load_config(config_dir)
label_key = config['data_kwargs']['label_key']
adata = sc.read_h5ad(adata_dir)
write_config(setting_config(adata, config), config_dir)

print("Original annote:")
print(adata.obs[label_key].unique())

tissue = 'Immune system'
sctype_res = search_sctype_db(
    cell_type='',
    tissue_type=tissue,
)


cm_res = search_cellmarker_db(
    cell_types=['B cell','Megakaryocyte','CD8 T','Natural killer','Monocyte','Myeloid','CD4 T','Dendritic'],
    tissue_type='Peripheral blood',
    cancer_type='Normal',
)

data_specific = {**sctype_res, **cm_res}
data_specific = clearn_prior_dict(data_specific, adata, min_genes = 5)
backgrounds = clearn_prior_dict(basis, adata, min_genes = 10)

print("#####")
print(adata.shape)
print(len(data_specific), len(backgrounds))

adata.uns[config['data_kwargs']['annotation_key']] = {
    'background': backgrounds,
    'clusters': data_specific
}
adata.uns['running_config'] = config
adata.write_h5ad(adata_dir)

Original annote:
['0', '2', '1', '3', '5', '4']
Categories (6, object): ['0', '1', '2', '3', '4', '5']
Loading cell type database
Searching for tissue type Immune system
Searching for cell type 
Searching for cell markers with tissue type Peripheral blood and cancer type Normal
#####
(1803, 1987)
14 270


### Bassez

In [26]:
data_name = 'Bassez'
project_dir = os.path.join(save_dir, data_name)

adata_dir = os.path.join(project_dir, 'adata.h5ad')
config_dir = os.path.join(project_dir, 'running_config.yaml')
config = load_config(config_dir)
label_key = config['data_kwargs']['label_key']
adata = sc.read_h5ad(adata_dir)
write_config(setting_config(adata, config), config_dir)

print("Original annote:")
print(adata.obs[label_key].unique())

cm_res = search_cellmarker_db(
    cell_types=['B cells','Cancer','Endothelial','Fibroblast','Mast','Myeloid','T cell','DC'],
    tissue_type='Breast',
    cancer_type='Breast Cancer',
)

data_specific = { **cm_res}
data_specific = clearn_prior_dict(data_specific, adata, min_genes = 5)
backgrounds = clearn_prior_dict(basis, adata, min_genes = 10)

print("#####")
print(adata.shape)
print(len(data_specific), len(backgrounds))

adata.uns[config['data_kwargs']['annotation_key']] = {
    'background': backgrounds,
    'clusters': data_specific
}
adata.uns['running_config'] = config
adata.write_h5ad(adata_dir)

Original annote:
['T_cell', 'Fibroblast', 'Endothelial_cell', 'B_cell', 'Cancer_cell', 'Myeloid_cell', 'pDC', 'Mast_cell']
Categories (8, object): ['B_cell', 'Cancer_cell', 'Endothelial_cell', 'Fibroblast', 'Mast_cell', 'Myeloid_cell', 'T_cell', 'pDC']
Searching for cell markers with tissue type Breast and cancer type Breast Cancer


/home/u12219016/jupyterlab/deepbicluster/evaluate/../src/utils/prior.py:125: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  markers = pd.Series(markers).dropna().unique()


#####
(144160, 2000)
5 247


### Mariana_24NC

In [27]:
data_name = 'Mariana_24NC'
project_dir = os.path.join(save_dir, data_name)

adata_dir = os.path.join(project_dir, 'adata.h5ad')
config_dir = os.path.join(project_dir, 'running_config.yaml')
config = load_config(config_dir)
label_key = config['data_kwargs']['label_key']
adata = sc.read_h5ad(adata_dir)
write_config(setting_config(adata, config), config_dir)

print("Original annote:")
print(adata.obs[label_key].unique().tolist())

adata.var = adata.var.reset_index().set_index('feature_name')
adata.var_names = adata.var_names.astype(str)

tissue = 'Immune system'
sctype_res = search_sctype_db(
    cell_type='',
    tissue_type=tissue,
)

cm_res = search_cellmarker_db(
    cell_types=[i for i in adata.obs[label_key].unique().tolist()],
    tissue_type='Peripheral blood',
    cancer_type='Normal',
)

data_specific = {**cm_res, **sctype_res}
data_specific = clearn_prior_dict(data_specific, adata, min_genes = 5)
backgrounds = clearn_prior_dict(basis, adata, min_genes = 10)

print("#####")
print(adata.shape)
print(len(data_specific), len(backgrounds))

data_specific.keys()

adata.uns[config['data_kwargs']['annotation_key']] = {
    'background': backgrounds,
    'clusters': data_specific
}
adata.uns['running_config'] = config
adata.write_h5ad(adata_dir)

Original annote:
['endothelial cell', 'epithelial cell', 'mononuclear phagocyte', 'plasmacytoid dendritic cell', 'B cell', 'fibroblast', 'T cell', 'mast cell', 'megakaryocyte', 'neutrophil', 'malignant cell']
Loading cell type database
Searching for tissue type Immune system
Searching for cell type 
Searching for cell markers with tissue type Peripheral blood and cancer type Normal


/home/u12219016/jupyterlab/deepbicluster/evaluate/../src/utils/prior.py:125: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  markers = pd.Series(markers).dropna().unique()


#####
(375739, 2000)
27 199


### GSE262072_CART_Processed_compress

In [28]:
data_name = 'GSE262072_CART_Processed_compress'
project_dir = os.path.join(save_dir, data_name)

adata_dir = os.path.join(project_dir, 'adata.h5ad')
config_dir = os.path.join(project_dir, 'running_config.yaml')
config = load_config(config_dir)
label_key = config['data_kwargs']['label_key']
adata = sc.read_h5ad(adata_dir)
write_config(setting_config(adata, config), config_dir)

print("Original annote:")
print(adata.obs[label_key].unique().tolist())

tissue = 'Immune system'
sctype_res = search_sctype_db(
    cell_type='',
    tissue_type=tissue,
)
sctype_res = filter_prior_dict(sctype_res, keywords=['T'])

cm_res = cellmarker_db.query("cancer_type != 'Normal'")
target_cell_types = [i for i in cm_res['cell_name'].unique() if 'CD8' in i]
cm_res = cm_res.query("cell_name in @target_cell_types")
cm_res = cm_res.groupby("cell_name").apply(lambda x: x['marker'].values.tolist()).to_dict()

data_specific = {**cm_res, **sctype_res}
data_specific = clearn_prior_dict(data_specific, adata, min_genes = 5)
backgrounds = clearn_prior_dict(basis, adata, min_genes = 10)

print("#####")
print(adata.shape)
print(len(data_specific), len(backgrounds))

data_specific.keys()

adata.uns[config['data_kwargs']['annotation_key']] = {
    'background': backgrounds,
    'clusters': data_specific
}
adata.uns['running_config'] = config
adata.write_h5ad(adata_dir)

Original annote:
['CARTQ', 'CARTR', 'CARTO', 'CARTP', 'CARTS', 'CARTT', 'CARTU', 'CARTV', 'CARTW', 'CARTX', 'CARTY', 'CARTZ', 'CART1', 'CART2', 'CART3', 'CART4', 'CART5', 'CART6', 'CART7', 'CART8', 'CART9', 'CART10', 'CARTI', 'CARTJ', 'CARTM', 'CARTN', 'CARTK', 'CARTL', 'CARTE', 'CARTF', 'CARTC', 'CARTD', 'CART11', 'CART12', 'CARTG', 'CARTH', 'CART13', 'CART14', 'CARTA', 'CARTB', 'CART15', 'CART16', 'CART17', 'CART18']
Loading cell type database
Searching for tissue type Immune system
Searching for cell type 
#####
(598063, 2000)
15 257


### Pancan_T_atals_integrated_full

In [29]:
data_name = 'Pancan_T_atals_integrated_full'
project_dir = os.path.join(save_dir, data_name)

adata_dir = os.path.join(project_dir, 'adata.h5ad')
config_dir = os.path.join(project_dir, 'running_config.yaml')
config = load_config(config_dir)
label_key = config['data_kwargs']['label_key']
adata = sc.read_h5ad(adata_dir)
write_config(setting_config(adata, config), config_dir)

print("Original annote:")
print(adata.obs[label_key].unique().tolist())

tissue = 'Immune system'
sctype_res = search_sctype_db(
    cell_type='',
    tissue_type=tissue,
)
sctype_res = filter_prior_dict(sctype_res, keywords=['T'])

cm_res = cellmarker_db.query("cancer_type != 'Normal'")
target_cell_types = [i for i in cm_res['cell_name'].unique() if ('CD8' in i) or ('CD4' in i)]
cm_res = cm_res.query("cell_name in @target_cell_types")
cm_res = cm_res.groupby("cell_name").apply(lambda x: x['marker'].values.tolist()).to_dict()

data_specific = {**cm_res, **sctype_res}
data_specific = clearn_prior_dict(data_specific, adata, min_genes = 5)
backgrounds = clearn_prior_dict(basis, adata, min_genes = 10)

print("#####")
print(adata.shape)
print(len(data_specific), len(backgrounds))

data_specific.keys()

adata.uns[config['data_kwargs']['annotation_key']] = {
    'background': backgrounds,
    'clusters': data_specific
}
adata.uns['running_config'] = config
adata.write_h5ad(adata_dir)

Original annote:
['CD4.c06.Tm.ANXA1', 'CD4.c10.Tm.CAPG', 'CD4.c15.Th17.IL23R', 'CD4.c14.Th17.SLC4A10', 'CD4.c13.Temra.CX3CR1', 'CD4.c01.Tn.TCF7', 'CD4.c07.Tm.ANXA2', 'CD4.c23.Mix.NME1', 'CD4.c24.Mix.NME2', 'CD4.c05.Tm.TNF', 'CD4.c19.Treg.S1PR1', 'CD4.c03.Tn.ADSL', 'CD4.c04.Tn.il7r', 'CD4.c22.ISG.IFIT1', 'CD4.c18.Treg.RTKN2', 'CD4.c20.Treg.TNFRSF9', 'CD4.c11.Tm.GZMA', 'CD4.c12.Tem.GZMK', 'CD4.c08.Tm.CREM', 'CD4.c16.Tfh.CXCR5', 'CD4.c02.Tn.PASK', 'CD4.c09.Tm.CCL5', 'CD4.c17.TfhTh1.CXCL13', 'CD4.c21.Treg.OAS1', 'CD8.c06.Tem.GZMK', 'CD8.c08.Tk.TYROBP', 'CD8.c07.Temra.CX3CR1', 'CD8.c01.Tn.MAL', 'CD8.c05.Tem.CXCR5', 'CD8.c02.Tm.IL7R', 'CD8.c10.Trm.ZNF683', 'CD8.c03.Tm.RPS12', 'CD8.c17.Tm.NME1', 'CD8.c16.MAIT.SLC4A10', 'CD8.c04.Tm.CD52', 'CD8.c15.ISG.IFIT1', 'CD8.c14.Tex.TCF7', 'CD8.c11.Tex.PDCD1', 'CD8.c12.Tex.CXCL13', 'CD8.c09.Tk.KIR2DL4', 'CD8.c13.Tex.myl12a']
Loading cell type database
Searching for tissue type Immune system
Searching for cell type 
#####
(315218, 2000)
18 248


### Lung_atlas

In [30]:
data_name = 'Lung_atlas'
project_dir = os.path.join(save_dir, data_name)

adata_dir = os.path.join(project_dir, 'adata.h5ad')
config_dir = os.path.join(project_dir, 'running_config.yaml')
config = load_config(config_dir)
label_key = config['data_kwargs']['label_key']
adata = sc.read_h5ad(adata_dir)
write_config(setting_config(adata, config), config_dir)

print("Original annote:")
print(adata.obs[label_key].unique().tolist())

tissue = 'Lung'
sctype_res = search_sctype_db(
    cell_type='',
    tissue_type=tissue,
)
# sctype_res = filter_prior_dict(sctype_res, keywords=['T'])
sctype_res.keys()

cm_res = search_cellmarker_db(
    cell_types=[
        'Secretory', 'TNK', 'Mast', 'B cell', 
        'Ciliated', 'Ionocytes', 'Basal', 'Dendritic', 
        'Neutrophil', 'Macrophage', 'Endothelium', 'Fibroblast', 'Lymphatic'],
    tissue_type='Lung',
    cancer_type='Normal',
)

data_specific = {**cm_res, **sctype_res}
data_specific = clearn_prior_dict(data_specific, adata, min_genes = 5)
backgrounds = clearn_prior_dict(basis, adata, min_genes = 10)

print("#####")
print(adata.shape)
print(len(data_specific), len(backgrounds))

data_specific.keys()

adata.uns[config['data_kwargs']['annotation_key']] = {
    'background': backgrounds,
    'clusters': data_specific
}
adata.uns['running_config'] = config
adata.write_h5ad(adata_dir)

Original annote:
['Secretory', 'T/NK cell', 'Mast cell', 'B cell', 'Ciliated', 'Ionocytes', 'Basal 2', 'Basal 1', 'Dendritic cell', 'Neutrophil_CD14_high', 'Neutrophils_IL1R2', 'Macrophage', 'Endothelium', 'Fibroblast', 'Lymphatic', 'Type 2']
Loading cell type database
Searching for tissue type Lung
Searching for cell type 
Searching for cell markers with tissue type Lung and cancer type Normal


/home/u12219016/jupyterlab/deepbicluster/evaluate/../src/utils/prior.py:125: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  markers = pd.Series(markers).dropna().unique()


#####
(21952, 2000)
21 250


## HLCA_subsampled

In [31]:
save_dir = '../results/subsampled/'
os.listdir(save_dir)

['HLCA_T_l3',
 'HLCA_Myeloid_l2',
 'HLCA_Secretory_lf',
 'HLCA_DC_l3',
 'HLCA_Basal_l3',
 'HLCA_Myeloid_lf',
 'HLCA_Fibroblasts_l3',
 'HLCA_Lymphoid_lf']

In [32]:
data_name = 'HLCA_T_l3'
project_dir = os.path.join(save_dir, data_name)

adata_dir = os.path.join(project_dir, 'adata.h5ad')
config_dir = os.path.join(project_dir, 'running_config.yaml')
config = load_config(config_dir)
label_key = config['data_kwargs']['label_key']
adata = sc.read_h5ad(adata_dir)
write_config(setting_config(adata, config), config_dir)

adata.var_names = adata.var['feature_name'].astype(str)
adata.var_names.name = 'gene symbol' # for save h5ad

print("Original annote:")
print(adata.obs[label_key].unique().tolist())

cm_res = cellmarker_db.query("cancer_type == 'Normal' & tissue_class == 'Lung'")
target_cell_types = [i for i in cm_res['cell_name'].unique() if ('T' in i)]
cm_res = cm_res.query("cell_name in @target_cell_types")
cm_res = cm_res.groupby("cell_name").apply(lambda x: x['marker'].values.tolist()).to_dict()

data_specific = {**cm_res}
data_specific = clearn_prior_dict(data_specific, adata, min_genes = 5)
backgrounds = clearn_prior_dict(basis, adata, min_genes = 10)

print("#####")
print(adata.shape)
print(len(data_specific), len(backgrounds))

data_specific.keys()

adata.uns[config['data_kwargs']['annotation_key']] = {
    'background': backgrounds,
    'clusters': data_specific
}
adata.uns['running_config'] = config
adata.write_h5ad(adata_dir)

Original annote:
['CD4 T cells', 'CD8 T cells', 'T cells proliferating']
#####
(1489, 2000)
5 449


In [40]:
data_name = 'HLCA_Myeloid_l2'
project_dir = os.path.join(save_dir, data_name)

adata_dir = os.path.join(project_dir, 'adata.h5ad')
config_dir = os.path.join(project_dir, 'running_config.yaml')
config = load_config(config_dir)
label_key = config['data_kwargs']['label_key']
adata = sc.read_h5ad(adata_dir)
write_config(setting_config(adata, config), config_dir)

adata.var_names = adata.var['feature_name'].astype(str)
adata.var_names.name = 'gene symbol' # for save h5ad

print("Original annote:")
print(adata.obs[label_key].unique().tolist())

sctype_res = filter_prior_dict(search_sctype_db(''), keywords=['Dendritic', 'Macrophages', 'Mast', 'Monocytes'])

# cm_res = cellmarker_db.query("cancer_type == 'Normal' & tissue_class == 'Lung'")
# target_cell_types = [i for i in cm_res['cell_name'].unique() if ('Myeloid' in i)]
# cm_res = cm_res.query("cell_name in @target_cell_types")
# cm_res = cm_res.groupby("cell_name").apply(lambda x: x['marker'].values.tolist()).to_dict()
cm_res = search_cellmarker_db(
    cell_types=['Dendritic', 'Macrophages', 'Mast', 'Monocytes'],
    tissue_type='Lung',
    cancer_type='Normal',
)

data_specific = {**cm_res,**sctype_res}
data_specific = clearn_prior_dict(data_specific, adata, min_genes = 5)
backgrounds = clearn_prior_dict(basis, adata, min_genes = 10)

print("#####")
print(adata.shape)
print(len(data_specific), len(backgrounds))

data_specific.keys()

adata.uns[config['data_kwargs']['annotation_key']] = {
    'background': backgrounds,
    'clusters': data_specific
}
adata.uns['running_config'] = config
adata.write_h5ad(adata_dir)

Original annote:
['Dendritic cells', 'Macrophages', 'Mast cells', 'Monocytes']
Loading cell type database
Searching for cell type 
Searching for cell markers with tissue type Lung and cancer type Normal


/home/u12219016/jupyterlab/deepbicluster/evaluate/../src/utils/prior.py:125: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  markers = pd.Series(markers).dropna().unique()


#####
(4868, 2000)
7 293


In [34]:
data_name = 'HLCA_Secretory_lf'
project_dir = os.path.join(save_dir, data_name)

adata_dir = os.path.join(project_dir, 'adata.h5ad')
config_dir = os.path.join(project_dir, 'running_config.yaml')
config = load_config(config_dir)
label_key = config['data_kwargs']['label_key']
adata = sc.read_h5ad(adata_dir)
write_config(setting_config(adata, config), config_dir)

adata.var_names = adata.var['feature_name'].astype(str)
adata.var_names.name = 'gene symbol' # for save h5ad

print("Original annote:")
print(adata.obs[label_key].unique().tolist())

sctype_res = filter_prior_dict(search_sctype_db('', tissue_type='Lung'), keywords=['Secretory', 'Airway'])

# cm_res = cellmarker_db.query("cancer_type == 'Normal' & tissue_class == 'Lung'")
# target_cell_types = [i for i in cm_res['cell_name'].unique() if ('Myeloid' in i)]
# cm_res = cm_res.query("cell_name in @target_cell_types")
# cm_res = cm_res.groupby("cell_name").apply(lambda x: x['marker'].values.tolist()).to_dict()
cm_res = search_cellmarker_db(
    cell_types=['Airway secretory cell', 'Secretory cell'],
    tissue_type='Lung',
    cancer_type='Normal',
)

data_specific = {**cm_res,**sctype_res}
data_specific = clearn_prior_dict(data_specific, adata, min_genes = 3)
backgrounds = clearn_prior_dict(basis, adata, min_genes = 10)

print("#####")
print(adata.shape)
print(len(data_specific), len(backgrounds))

data_specific.keys()

adata.uns[config['data_kwargs']['annotation_key']] = {
    'background': backgrounds,
    'clusters': data_specific
}
adata.uns['running_config'] = config
adata.write_h5ad(adata_dir)

Original annote:
['Goblet (nasal)', 'Club (nasal)', 'Club (non-nasal)', 'pre-TB secretory', 'AT0', 'Goblet (bronchial)']
Loading cell type database
Searching for tissue type Lung
Searching for cell type 
Searching for cell markers with tissue type Lung and cancer type Normal
#####
(4214, 2000)
3 283


In [35]:
data_name = 'HLCA_DC_l3'
project_dir = os.path.join(save_dir, data_name)

adata_dir = os.path.join(project_dir, 'adata.h5ad')
config_dir = os.path.join(project_dir, 'running_config.yaml')
config = load_config(config_dir)
label_key = config['data_kwargs']['label_key']
adata = sc.read_h5ad(adata_dir)
write_config(setting_config(adata, config), config_dir)

adata.var_names = adata.var['feature_name'].astype(str)
adata.var_names.name = 'gene symbol' # for save h5ad

print("Original annote:")
print(adata.obs[label_key].unique().tolist())

sctype_res = filter_prior_dict(search_sctype_db('', tissue_type='Lung'), keywords=['Dendritic'])

cm_res = cellmarker_db.query("cancer_type == 'Normal' & tissue_class == 'Lung'")
target_cell_types = [i for i in cm_res['cell_name'].unique() if ('dendritic' in i.lower())]
cm_res = cm_res.query("cell_name in @target_cell_types")
cm_res = cm_res.groupby("cell_name").apply(lambda x: x['marker'].values.tolist()).to_dict()

data_specific = {**cm_res,**sctype_res}
data_specific = clearn_prior_dict(data_specific, adata, min_genes = 3)
backgrounds = clearn_prior_dict(basis, adata, min_genes = 10)

print("#####")
print(adata.shape)
print(len(data_specific), len(backgrounds))

data_specific.keys()

adata.uns[config['data_kwargs']['annotation_key']] = {
    'background': backgrounds,
    'clusters': data_specific
}
adata.uns['running_config'] = config
adata.write_h5ad(adata_dir)

Original annote:
['DC1', 'DC2', 'Migratory DCs', 'Plasmacytoid DCs']
Loading cell type database
Searching for tissue type Lung
Searching for cell type 
#####
(1187, 2000)
5 305


In [53]:
data_name = 'HLCA_Basal_l3'
project_dir = os.path.join(save_dir, data_name)

adata_dir = os.path.join(project_dir, 'adata.h5ad')
config_dir = os.path.join(project_dir, 'running_config.yaml')
config = load_config(config_dir)
label_key = config['data_kwargs']['label_key']
adata = sc.read_h5ad(adata_dir)
write_config(setting_config(adata, config), config_dir)

adata.var_names = adata.var['feature_name'].astype(str)
adata.var_names.name = 'gene symbol' # for save h5ad

print("Original annote:")
print(adata.obs[label_key].unique().tolist())

sctype_res = filter_prior_dict(search_sctype_db(''), keywords=['Basal'])

cm_res = cellmarker_db.query("cancer_type == 'Normal' & tissue_class == 'Lung'")
target_cell_types = [i for i in cm_res['cell_name'].unique() if ('basal' in i.lower())]
cm_res = cm_res.query("cell_name in @target_cell_types")
cm_res = cm_res.groupby("cell_name").apply(lambda x: x['marker'].values.tolist()).to_dict()

data_specific = {**cm_res,**sctype_res}
data_specific = clearn_prior_dict(data_specific, adata, min_genes = 3)
backgrounds = clearn_prior_dict(basis, adata, min_genes = 10)

print("#####")
print(adata.shape)
print(len(data_specific), len(backgrounds))

data_specific.keys()

adata.uns[config['data_kwargs']['annotation_key']] = {
    'background': backgrounds,
    'clusters': data_specific
}
adata.uns['running_config'] = config
adata.write_h5ad(adata_dir)

Original annote:
['Basal resting', 'Hillock-like', 'Suprabasal']
Loading cell type database
Searching for cell type 
#####
(4918, 2000)
2 353


In [54]:
data_name = 'HLCA_Myeloid_lf'
project_dir = os.path.join(save_dir, data_name)

adata_dir = os.path.join(project_dir, 'adata.h5ad')
config_dir = os.path.join(project_dir, 'running_config.yaml')
config = load_config(config_dir)
label_key = config['data_kwargs']['label_key']
adata = sc.read_h5ad(adata_dir)
write_config(setting_config(adata, config), config_dir)

adata.var_names = adata.var['feature_name'].astype(str)
adata.var_names.name = 'gene symbol' # for save h5ad

print("Original annote:")
print(adata.obs[label_key].unique().tolist())

sctype_res = filter_prior_dict(search_sctype_db('',), keywords=['Dendritic', 'Macrophages', 'Mast', 'Monocytes', 'DC'])

# cm_res = cellmarker_db.query("cancer_type == 'Normal' & tissue_class == 'Lung'")
# target_cell_types = [i for i in cm_res['cell_name'].unique() if ('myeloid' in i.lower())]
# cm_res = cm_res.query("cell_name in @target_cell_types")
# cm_res = cm_res.groupby("cell_name").apply(lambda x: x['marker'].values.tolist()).to_dict()
cm_res = search_cellmarker_db(
    cell_types=[
        'Alveolar macrophage', 'monocytes', 'Monocyte-derived cell', 'Non-classical monocytes', 'DC2', 
        'Interstitial macrophage', 'Mast', 'cDC', 'pDC', 'DC1'],
    tissue_type='Lung',
    cancer_type='Normal',
)

data_specific = {**cm_res,**sctype_res}
data_specific = clearn_prior_dict(data_specific, adata, min_genes = 3)
backgrounds = clearn_prior_dict(basis, adata, min_genes = 10)

print("#####")
print(adata.shape)
print(len(data_specific), len(backgrounds))

data_specific.keys()

adata.uns[config['data_kwargs']['annotation_key']] = {
    'background': backgrounds,
    'clusters': data_specific
}
adata.uns['running_config'] = config
adata.write_h5ad(adata_dir)

Original annote:
['Alveolar Mph MT-positive', 'Alveolar macrophages', 'Classical monocytes', 'Monocyte-derived Mph', 'Non-classical monocytes', 'DC2', 'Interstitial Mph perivascular', 'Mast cells', 'Alveolar Mph CCL3+', 'Alveolar Mph proliferating', 'Migratory DCs', 'Plasmacytoid DCs', 'DC1']
Loading cell type database
Searching for cell type 
Searching for cell markers with tissue type Lung and cancer type Normal


/home/u12219016/jupyterlab/deepbicluster/evaluate/../src/utils/prior.py:125: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  markers = pd.Series(markers).dropna().unique()


#####
(4905, 2000)
8 276


In [59]:
data_name = 'HLCA_Fibroblasts_l3'
project_dir = os.path.join(save_dir, data_name)

adata_dir = os.path.join(project_dir, 'adata.h5ad')
config_dir = os.path.join(project_dir, 'running_config.yaml')
config = load_config(config_dir)
label_key = config['data_kwargs']['label_key']
adata = sc.read_h5ad(adata_dir)
write_config(setting_config(adata, config), config_dir)

adata.var_names = adata.var['feature_name'].astype(str)
adata.var_names.name = 'gene symbol' # for save h5ad

print("Original annote:")
print(adata.obs[label_key].unique().tolist())

sctype_res = filter_prior_dict(search_sctype_db(''), keywords=['fibroblasts','Pericytes',])

# cm_res = cellmarker_db.query("cancer_type == 'Normal' & tissue_class == 'Lung'")
# target_cell_types = [i for i in cm_res['cell_name'].unique() if ('myeloid' in i.lower())]
# cm_res = cm_res.query("cell_name in @target_cell_types")
# cm_res = cm_res.groupby("cell_name").apply(lambda x: x['marker'].values.tolist()).to_dict()
cm_res = search_cellmarker_db(
    cell_types=['Myofibroblast', 'Fibroblast', 'Pericyte','Subpleural'],
    tissue_type='Lung',
    cancer_type='Normal',
)

data_specific = {**cm_res,**sctype_res}
data_specific = clearn_prior_dict(data_specific, adata, min_genes = 3)
backgrounds = clearn_prior_dict(basis, adata, min_genes = 10)

print("#####")
print(adata.shape)
print(len(data_specific), len(backgrounds))

data_specific.keys()

adata.uns[config['data_kwargs']['annotation_key']] = {
    'background': backgrounds,
    'clusters': data_specific
}
adata.uns['running_config'] = config
adata.write_h5ad(adata_dir)

Original annote:
['Adventitial fibroblasts', 'Alveolar fibroblasts', 'Pericytes', 'Peribronchial fibroblasts', 'Subpleural fibroblasts']
Loading cell type database
Searching for cell type 
Searching for cell markers with tissue type Lung and cancer type Normal


/home/u12219016/jupyterlab/deepbicluster/evaluate/../src/utils/prior.py:125: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  markers = pd.Series(markers).dropna().unique()


#####
(4879, 2000)
2 284


In [39]:
data_name = 'HLCA_Lymphoid_lf'
project_dir = os.path.join(save_dir, data_name)

adata_dir = os.path.join(project_dir, 'adata.h5ad')
config_dir = os.path.join(project_dir, 'running_config.yaml')
config = load_config(config_dir)
label_key = config['data_kwargs']['label_key']
adata = sc.read_h5ad(adata_dir)
write_config(setting_config(adata, config), config_dir)

adata.var_names = adata.var['feature_name'].astype(str)
adata.var_names.name = 'gene symbol' # for save h5ad

print("Original annote:")
print(adata.obs[label_key].unique().tolist())

sctype_res = filter_prior_dict(search_sctype_db(''), keywords=['NK', 'CD4', 'CD8', 'B cell', 'Plasma'])
sctype_res.keys()

# cm_res = cellmarker_db.query("cancer_type == 'Normal' & tissue_class == 'Lung'")
# target_cell_types = [i for i in cm_res['cell_name'].unique() if ('myeloid' in i.lower())]
# cm_res = cm_res.query("cell_name in @target_cell_types")
# cm_res = cm_res.groupby("cell_name").apply(lambda x: x['marker'].values.tolist()).to_dict()
cm_res = search_cellmarker_db(
    cell_types=['NK', 'CD4', 'CD8', 'B cell', 'Plasma'],
    tissue_type='Lung',
    cancer_type='Normal',
)

data_specific = {**cm_res,**sctype_res}
data_specific = clearn_prior_dict(data_specific, adata, min_genes = 3)
backgrounds = clearn_prior_dict(basis, adata, min_genes = 10)

print("#####")
print(adata.shape)
print(len(data_specific), len(backgrounds))

data_specific.keys()

adata.uns[config['data_kwargs']['annotation_key']] = {
    'background': backgrounds,
    'clusters': data_specific
}
adata.uns['running_config'] = config
adata.write_h5ad(adata_dir)

Original annote:
['NK cells', 'CD4 T cells', 'CD8 T cells', 'T cells proliferating', 'B cells', 'Plasma cells']
Loading cell type database
Searching for cell type 
Searching for cell markers with tissue type Lung and cancer type Normal


/home/u12219016/jupyterlab/deepbicluster/evaluate/../src/utils/prior.py:125: FutureWarning: The default dtype for empty Series will be 'object' instead of 'float64' in a future version. Specify a dtype explicitly to silence this warning.
  markers = pd.Series(markers).dropna().unique()


#####
(4824, 2000)
18 287
